<a href="https://colab.research.google.com/github/santiagonajera/Ejemplo-RAG-Contratos/blob/main/RAG_analisis_contrato.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG para analizar un contrato (Word)

Pipeline 100% local y gratuito (sin API keys):

1. Subís el contrato en `.docx` o `.doc`
2. Se extrae y trocea (`chunking`) el texto
3. Se generan embeddings con `sentence-transformers` (modelo multilingüe)
4. Se indexan los chunks con `FAISS` (búsqueda por similitud)
5. Ante cada pregunta, se recuperan los chunks más relevantes y se arma un prompt
6. Un modelo de HuggingFace (local, corre en el Colab) genera la respuesta usando solo esos chunks como contexto

> Modelo de generación por defecto: `Qwen/Qwen2.5-1.5B-Instruct` (multilingüe, liviano). Si no tenés GPU en el runtime, andá a `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)` para que corra más rápido. También funciona en CPU, solo que más lento.

## 1. Instalar dependencias

Incluye `libreoffice` porque algunos contratos vienen en `.doc` (formato binario viejo) en vez de `.docx`, y hace falta convertirlos.

In [3]:
!pip install -q python-docx sentence-transformers faiss-cpu transformers accelerate
!apt-get -qq update --allow-releaseinfo-change -o Acquire::Retries=3
!apt-get -qq install -y --fix-missing libreoffice-writer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package poppler-data.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../00-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package libtext-iconv-perl.
Preparing to unpack .../01-libtext-iconv-perl_1.7-7build3_amd64.deb ...
Unpacking libtext-iconv-perl (1.7-7build3) ...
Selecting previously unselected package apparmor.
Preparing to unpack .../02-apparmor_3.0.4-2ubuntu2.5_amd64.deb ...
Unpacking apparmor (3.0.4-2ubuntu2.5) ...
Selecting previousl

## 2. Subir el contrato (.docx o .doc)

In [4]:
from google.colab import files

uploaded = files.upload()  # seleccioná tu archivo .docx o .doc
archivo_original = list(uploaded.keys())[0]
print(f"Archivo cargado: {archivo_original}")

Saving contrato-privado-de-compraventac7.doc to contrato-privado-de-compraventac7 (1).doc
Archivo cargado: contrato-privado-de-compraventac7 (1).doc


## 3. Convertir a .docx si hace falta

`python-docx` solo lee `.docx` (es un zip con XML adentro). Si el archivo es `.doc` (formato binario viejo), lo convertimos con LibreOffice en modo headless.

In [5]:
import os
import subprocess

def asegurar_docx(path):
    nombre, ext = os.path.splitext(path)
    if ext.lower() == ".docx":
        return path

    print(f"Convirtiendo {path} a .docx con LibreOffice...")
    resultado = subprocess.run(
        ["libreoffice", "--headless", "--convert-to", "docx", "--outdir", ".", path],
        capture_output=True, text=True
    )
    print(resultado.stdout)
    if resultado.returncode != 0:
        raise RuntimeError(f"Error al convertir: {resultado.stderr}")

    docx_generado = nombre + ".docx"
    if not os.path.exists(docx_generado):
        raise FileNotFoundError(f"No se encontró el archivo convertido: {docx_generado}")

    return docx_generado

docx_path = asegurar_docx(archivo_original)
print(f"Archivo listo para procesar: {docx_path}")

Convirtiendo contrato-privado-de-compraventac7 (1).doc a .docx con LibreOffice...
convert /content/contrato-privado-de-compraventac7 (1).doc -> /content/contrato-privado-de-compraventac7 (1).docx using filter : Office Open XML Text

Archivo listo para procesar: contrato-privado-de-compraventac7 (1).docx


## 4. Extraer texto del Word

In [6]:
import docx

def extraer_texto_docx(path):
    doc = docx.Document(path)
    partes = []

    # Párrafos
    for p in doc.paragraphs:
        if p.text.strip():
            partes.append(p.text.strip())

    # Tablas (muchos contratos tienen cláusulas o anexos en tablas)
    for tabla in doc.tables:
        for fila in tabla.rows:
            celdas = [c.text.strip() for c in fila.cells if c.text.strip()]
            if celdas:
                partes.append(" | ".join(celdas))

    return "\n".join(partes)

texto_contrato = extraer_texto_docx(docx_path)
print(f"Longitud del texto: {len(texto_contrato)} caracteres")
print(texto_contrato[:800])

Longitud del texto: 5968 caracteres
CONTRATO PRIVADO DE COMPRAVENTA
CONTRATO PRIVADO DE COMPRAVENTA QUE CELEBRAN POR UNA PARTE                    A QUIENES EN LO SUCESIVO SE LES  DENOMINARA  COMO  "LA PARTE  VENDEDORA" Y DE OTRA PARTE                    ,  A QUIENES EN LO SUCESIVO SE LES DENOMINARA COMO "LA PARTE  COMPRADORA",  MISMO CONTRATO QUE SUJETAN A LO CONTENIDO EN LAS SIGUIENTES  DECLARACIONES Y CLAUSULAS.
DECLARACIONES
I.-  Declara "LA PARTE VENDEDORA"  bajo protesta de decir verdad:
a).- Ser legítimos propietarios del inmueble identificado como                    lo cual acreditan con el primer testimonio de la escritura pública número          otorgada ante la fé del señor licenciado      Notario Público número         de la ciudad de                    , el cual se encuentra debidamente inscrito en el Registro Pú


## 5. Chunking (trocear el contrato)

Se divide el texto en fragmentos con overlap para no perder contexto entre cláusulas.

In [7]:
def chunk_texto(texto, chunk_size=800, overlap=150):
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = inicio + chunk_size
        chunks.append(texto[inicio:fin])
        inicio += chunk_size - overlap
    return [c for c in chunks if c.strip()]

chunks = chunk_texto(texto_contrato)
print(f"Cantidad de chunks: {len(chunks)}")
print("\n--- Ejemplo de chunk ---\n")
print(chunks[0])

Cantidad de chunks: 10

--- Ejemplo de chunk ---

CONTRATO PRIVADO DE COMPRAVENTA
CONTRATO PRIVADO DE COMPRAVENTA QUE CELEBRAN POR UNA PARTE                    A QUIENES EN LO SUCESIVO SE LES  DENOMINARA  COMO  "LA PARTE  VENDEDORA" Y DE OTRA PARTE                    ,  A QUIENES EN LO SUCESIVO SE LES DENOMINARA COMO "LA PARTE  COMPRADORA",  MISMO CONTRATO QUE SUJETAN A LO CONTENIDO EN LAS SIGUIENTES  DECLARACIONES Y CLAUSULAS.
DECLARACIONES
I.-  Declara "LA PARTE VENDEDORA"  bajo protesta de decir verdad:
a).- Ser legítimos propietarios del inmueble identificado como                    lo cual acreditan con el primer testimonio de la escritura pública número          otorgada ante la fé del señor licenciado      Notario Público número         de la ciudad de                    , el cual se encuentra debidamente inscrito en el Registro Pú


## 6. Embeddings + índice FAISS

In [8]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modelo multilingüe liviano, corre bien en CPU
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embeddings = modelo_embeddings.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # producto interno = similitud coseno (vectores normalizados)
index.add(embeddings)

print(f"Índice FAISS creado con {index.ntotal} vectores de dimensión {dimension}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índice FAISS creado con 10 vectores de dimensión 384


## 7. Función de recuperación (retrieval)

In [9]:
def recuperar_chunks(pregunta, top_k=4):
    emb_pregunta = modelo_embeddings.encode([pregunta], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(emb_pregunta, top_k)
    resultados = [(chunks[i], float(scores[0][j])) for j, i in enumerate(indices[0])]
    return resultados

# prueba rápida
for chunk, score in recuperar_chunks("en que sitio se firma?"):
    print(f"[score {score:.3f}] {chunk[:200]}...\n")

[score 0.451]  señor licenciado      Notario Público número         de la ciudad de                    , el cual se encuentra debidamente inscrito en el Registro Público de la Propiedad de la ciudad de             ...

[score 0.375] islación civil en la materia.
NOVENA.- Para todo lo relacionado con la interpretación y cumplimiento del presente contrato las partes desean que se sujete a las leyes y tribunales de la ciudad de     ...

[score 0.370]  la firma de la escritura pública  se deberá cubrir el saldo del precio de la operación, mediante cheque certificado a nombre de "LA PARTE VENDEDORA", momento en que esta deberá entregar la propiedad ...

[score 0.343] ORA                        LA PARTE COMPRADORA
TESTIGO                                                         TESTIGO...



## 8. Modelo de generación (LLM local, HuggingFace)

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

NOMBRE_MODELO = "Qwen/Qwen2.5-1.5B-Instruct"  # multilingüe, liviano
# Alternativa aún más chica si el Colab se queda sin memoria: "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    NOMBRE_MODELO,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

generador = pipeline(
    "text-generation",
    model=modelo_llm,
    tokenizer=tokenizer,
    max_new_tokens=400,
    temperature=0.2,
    do_sample=True,
)

print("Modelo cargado. GPU disponible:", torch.cuda.is_available())

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo cargado. GPU disponible: False


## 9. Función RAG completa (retrieval + prompt + generación)

In [12]:
def responder_sobre_contrato(pregunta, top_k=4, mostrar_contexto=False):
    chunks_relevantes = recuperar_chunks(pregunta, top_k=top_k)
    contexto = "\n---\n".join(c for c, _ in chunks_relevantes)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Sos un asistente legal que analiza contratos. "
                "Respondé ÚNICAMENTE en base al CONTEXTO provisto. "
                "Si la información no está en el contexto, decí explícitamente "
                "'No encuentro esa información en el contrato'. "
                "Sé preciso y citá la cláusula o fragmento relevante cuando sea posible."
            ),
        },
        {
            "role": "user",
            "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}",
        },
    ]

    prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    salida = generador(prompt)[0]["generated_text"]
    respuesta = salida[len(prompt):].strip()

    if mostrar_contexto:
        print("=== CONTEXTO USADO ===")
        print(contexto)
        print("=======================\n")

    return respuesta

## 10. Probar con preguntas sobre el contrato

In [13]:
pregunta = "¿Cuáles son las partes que firman el contrato?"
print(responder_sobre_contrato(pregunta, mostrar_contexto=True))

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== CONTEXTO USADO ===
islación civil en la materia.
NOVENA.- Para todo lo relacionado con la interpretación y cumplimiento del presente contrato las partes desean que se sujete a las leyes y tribunales de la ciudad de                    .
DECIMA.- Para todos los efectos legales a que haya lugar las partes  señalan los domicilios que a continuación se mencionan
PARTE VENDEDORA
PARTE COMPRADORA
Ambas partes estando conformes con el contenido y clausulado del presente contrato lo firman el día                    de                    de 199   al margen en cada una de sus hojas y al calce en esta última para todos los efectos legales  a que haya lugar.
LA PARTE VENDEDORA                        LA PARTE COMPRADORA
TESTIGO                                                         TESTIGO
---
  con la señora              .
El señor                    , soltero, y que adquirió los derechos que le corresponden sobre el inmueble materia del presente contrato bajo este estado civil.
II.- Declara "

In [ ]:
pregunta = "¿Cuál es la duración del contrato y cómo se renueva?"
print(responder_sobre_contrato(pregunta))

In [ ]:
pregunta = "¿Qué penalidades existen por incumplimiento o terminación anticipada?"
print(responder_sobre_contrato(pregunta))

## 11. (Opcional) Checklist automático de análisis

Corre una tanda de preguntas típicas de revisión de contratos y arma un resumen.

In [ ]:
checklist = [
    "¿Quiénes son las partes del contrato?",
    "¿Cuál es el objeto/alcance del contrato?",
    "¿Cuál es la duración y las condiciones de renovación?",
    "¿Cuáles son las obligaciones principales de cada parte?",
    "¿Cuál es el monto, forma y plazos de pago?",
    "¿Qué causales de terminación anticipada existen?",
    "¿Hay cláusulas de penalidad o indemnización?",
    "¿Qué ley aplica y cuál es la jurisdicción en caso de disputa?",
    "¿Existen cláusulas de confidencialidad o no competencia?",
]

resumen = []
for pregunta in checklist:
    respuesta = responder_sobre_contrato(pregunta)
    resumen.append((pregunta, respuesta))
    print(f"### {pregunta}\n{respuesta}\n")

# resumen queda disponible como lista de tuplas (pregunta, respuesta)
# se puede volcar a un .docx/.txt si querés exportarlo

## Notas y opciones

- **Archivos .doc**: se convierten automáticamente a `.docx` con LibreOffice (celda 3). Si tu archivo ya es `.docx`, esa celda no hace nada (lo deja pasar tal cual).
- **Embeddings**: se usó `sentence-transformers` (gratis, local). Alternativa con más calidad pero con costo/API key: `OpenAIEmbeddings` (`text-embedding-3-small`).
- **Generación (LLM)**: se usó un modelo open-source de HuggingFace (gratis, corre en el Colab). Alternativas:
  - **Claude (Anthropic API)**: mejor calidad de análisis legal, requiere `pip install anthropic` y una API key (`ANTHROPIC_API_KEY`).
  - **OpenAI (GPT)**: requiere `pip install openai` y una API key.
  - Si más adelante querés migrar a alguna API, solo hay que reemplazar la función `responder_sobre_contrato` en la parte de generación — el retrieval (FAISS + embeddings) queda igual.
- **Vector store**: se usó FAISS en memoria (ideal para 1 contrato). Para múltiples documentos o persistencia entre sesiones, considerar Chroma o Qdrant.
- **Chunking**: el tamaño de chunk (800 caracteres) y overlap (150) son ajustables. Para contratos con cláusulas muy largas, conviene subir el `chunk_size`.